In [3]:

from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
from animal_shelter import AnimalShelter

import pandas as pd
import plotly.express as px
import base64
import os

username = "aacuser"
password = "TORi5576!"

shelter = AnimalShelter(username, password)

def get_query(rescue_type):
    if rescue_type == "Water":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    if rescue_type == "Mountain":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    if rescue_type == "Disaster":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    return {}


def get_data(rescue_type):
    records = list(shelter.read(get_query(rescue_type)))

    if not records:
        return []

    df = pd.DataFrame(records)

    if "_id" in df.columns:
        df = df.drop(columns=["_id"])

    for column in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[column]):
            df[column] = df[column].astype(str)

    df = df.astype(object).where(pd.notnull(df), None)

    return df.to_dict("records")


app = JupyterDash(__name__)

logo_path = "code_files/Grazioso Salvare Logo.png"

logo = None

if os.path.exists(logo_path):
    with open(logo_path, "rb") as image_file:
        encoded = base64.b64encode(
            image_file.read()
        ).decode()

    logo = html.A(
        html.Img(
            src="data:image/png;base64," + encoded,
            style={"height": "100px"}
        ),
        href="https://www.snhu.edu",
        target="_blank"
    )


app.layout = html.Div([

    html.Div(
        [logo] if logo else [],
        style={"textAlign": "center"}
    ),

    html.H1(
        "Grazioso Salvare Rescue Dashboard",
        style={"textAlign": "center"}
    ),

    html.H3(
        "Created by Shatoria Giles",
        style={"textAlign": "center"}
    ),

    html.Hr(),

    html.H3("Select Rescue Type"),

    dcc.RadioItems(
        id="rescue-filter",
        options=[
            {"label": "Water Rescue", "value": "Water"},
            {
                "label": "Mountain or Wilderness Rescue",
                "value": "Mountain"
            },
            {
                "label": "Disaster or Individual Tracking",
                "value": "Disaster"
            },
            {"label": "Reset", "value": "Reset"}
        ],
        value="Reset",
        labelStyle={
            "display": "inline-block",
            "marginRight": "20px"
        }
    ),

    html.Hr(),

    html.H3("Animal Shelter Data"),

    dash_table.DataTable(
        id="animal-table",
        data=[],
        columns=[],
        page_size=10,
        page_action="native",
        sort_action="native",
        filter_action="native",
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "padding": "10px",
            "minWidth": "100px"
        }
    ),

    html.Hr(),

    html.Div([

        html.Div([
            dcc.Graph(id="breed-chart")
        ], style={
            "width": "48%",
            "display": "inline-block",
            "verticalAlign": "top"
        }),

        html.Div([
            dcc.Graph(id="location-map")
        ], style={
            "width": "48%",
            "display": "inline-block",
            "verticalAlign": "top"
        })

    ])

], style={
    "padding": "20px",
    "fontFamily": "Arial"
})


@app.callback(
    [
        Output("animal-table", "data"),
        Output("animal-table", "columns"),
        Output("breed-chart", "figure"),
        Output("location-map", "figure")
    ],
    [Input("rescue-filter", "value")]
)
def update_dashboard(rescue_type):

    records = get_data(rescue_type)

    if not records:
        empty = px.scatter(
            title="No Matching Animal Records"
        )
        return [], [], empty, empty

    df = pd.DataFrame(records)

    columns = [
        {"name": col, "id": col}
        for col in df.columns
    ]

    if "breed" in df.columns:
        breed_counts = (
            df["breed"]
            .value_counts()
            .head(10)
            .reset_index()
        )
        breed_counts.columns = ["Breed", "Count"]

        pie_chart = px.pie(
            breed_counts,
            names="Breed",
            values="Count",
            title="Top 10 Dog Breeds"
        )
    else:
        pie_chart = px.pie(title="No Breed Data")

    if (
        "location_lat" in df.columns
        and "location_long" in df.columns
    ):
        map_df = df.copy()

        map_df["location_lat"] = pd.to_numeric(
            map_df["location_lat"],
            errors="coerce"
        )
        map_df["location_long"] = pd.to_numeric(
            map_df["location_long"],
            errors="coerce"
        )

        map_df = map_df.dropna(
            subset=["location_lat", "location_long"]
        )

        map_df = map_df[
            map_df["location_lat"].between(-90, 90)
            & map_df["location_long"].between(-180, 180)
        ]

        map_df = map_df.head(1000)

        if not map_df.empty:
            map_chart = px.scatter_mapbox(
                map_df,
                lat="location_lat",
                lon="location_long",
                hover_name=(
                    "name" if "name" in map_df.columns else None
                ),
                hover_data=(
                    ["breed"] if "breed" in map_df.columns else None
                ),
                zoom=9,
                height=450,
                title="Animal Locations"
            )

            map_chart.update_layout(
                mapbox_style="open-street-map",
                margin={"r": 0, "t": 40, "l": 0, "b": 0}
            )
        else:
            map_chart = px.scatter(
                title="No Valid Locations"
            )
    else:
        map_chart = px.scatter(
            title="Location Columns Not Found"
        )

    return records, columns, pie_chart, map_chart

app.run_server(mode="external", port=8050)

Dash app running on https://lunchpilot-collectmayor-3000.codio.io/proxy/8050/
